# Day 69 — Model Monitoring with Evidently AI
### Data drift detection, prediction drift, feature distribution monitoring, alerting

## 1. Setup

In [1]:
import sys
!{sys.executable} -m pip install evidently scikit-learn pandas numpy --quiet
print("Setup complete")

Setup complete



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Why Model Monitoring Matters

In [2]:
import pandas as pd
import numpy as np

drift_explanation = """
WHY MODELS DEGRADE IN PRODUCTION
==================================

A model trained on historical data makes an assumption:
  "The future will look like the past."

This assumption breaks when:

1. DATA DRIFT (Covariate Shift)
   The INPUT distribution changes after deployment.
   Example: Titanic model trained on 1912 passengers.
   If deployed for a modern cruise ship, passenger age/class
   distributions are completely different -- model sees
   inputs it was never trained on.

2. CONCEPT DRIFT (Label Drift)  
   The RELATIONSHIP between inputs and outputs changes.
   Example: A fraud detection model trained pre-pandemic.
   Spending patterns changed so dramatically that what looked
   like fraud before is now normal, and vice versa.

3. PREDICTION DRIFT
   The MODEL OUTPUT distribution changes.
   Symptom: model was predicting 60% class 1, now predicting 40%.
   Could indicate data drift OR concept drift.

4. FEATURE DRIFT
   Individual features change -- new missing values, different
   ranges, new categories appear.

WITHOUT MONITORING:
   Model degrades silently. Users notice bad predictions.
   Engineering team scrambles to find what changed.
   Could take weeks to diagnose.

WITH MONITORING:
   Alert fires when drift exceeds threshold.
   Team investigates immediately.
   Model retrained or rules adjusted before users notice.
"""
print(drift_explanation)


WHY MODELS DEGRADE IN PRODUCTION

A model trained on historical data makes an assumption:
  "The future will look like the past."

This assumption breaks when:

1. DATA DRIFT (Covariate Shift)
   The INPUT distribution changes after deployment.
   Example: Titanic model trained on 1912 passengers.
   If deployed for a modern cruise ship, passenger age/class
   distributions are completely different -- model sees
   inputs it was never trained on.

2. CONCEPT DRIFT (Label Drift)  
   The RELATIONSHIP between inputs and outputs changes.
   Example: A fraud detection model trained pre-pandemic.
   Spending patterns changed so dramatically that what looked
   like fraud before is now normal, and vice versa.

3. PREDICTION DRIFT
   The MODEL OUTPUT distribution changes.
   Symptom: model was predicting 60% class 1, now predicting 40%.
   Could indicate data drift OR concept drift.

4. FEATURE DRIFT
   Individual features change -- new missing values, different
   ranges, new categories app

## 3. Create Reference and Production Datasets

In [3]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

np.random.seed(42)

# simulate Titanic-like features
feature_names = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_enc", "Embarked_enc"]

# REFERENCE DATA -- what the model was trained on (historical)
n_ref = 500
reference_data = pd.DataFrame({
    "Pclass": np.random.choice([1, 2, 3], n_ref, p=[0.25, 0.30, 0.45]),
    "Age": np.random.normal(30, 12, n_ref).clip(1, 80),
    "SibSp": np.random.choice([0, 1, 2, 3], n_ref, p=[0.60, 0.25, 0.10, 0.05]),
    "Parch": np.random.choice([0, 1, 2], n_ref, p=[0.70, 0.20, 0.10]),
    "Fare": np.random.exponential(30, n_ref).clip(5, 300),
    "Sex_enc": np.random.choice([0, 1], n_ref, p=[0.65, 0.35]),
    "Embarked_enc": np.random.choice([0, 1, 2], n_ref, p=[0.70, 0.20, 0.10]),
})
reference_data["target"] = (
    (reference_data["Sex_enc"] == 1) * 0.4 +
    (reference_data["Pclass"] == 1) * 0.3 +
    np.random.normal(0, 0.1, n_ref)
).clip(0, 1).round().astype(int)

# CURRENT (PRODUCTION) DATA -- drifted distribution
n_curr = 200
current_data = pd.DataFrame({
    "Pclass": np.random.choice([1, 2, 3], n_curr, p=[0.50, 0.35, 0.15]),  # DRIFTED: more 1st class
    "Age": np.random.normal(45, 15, n_curr).clip(1, 80),                   # DRIFTED: older passengers
    "SibSp": np.random.choice([0, 1, 2, 3], n_curr, p=[0.60, 0.25, 0.10, 0.05]),
    "Parch": np.random.choice([0, 1, 2], n_curr, p=[0.70, 0.20, 0.10]),
    "Fare": np.random.exponential(60, n_curr).clip(5, 300),                 # DRIFTED: higher fares
    "Sex_enc": np.random.choice([0, 1], n_curr, p=[0.65, 0.35]),
    "Embarked_enc": np.random.choice([0, 1, 2], n_curr, p=[0.70, 0.20, 0.10]),
})
current_data["target"] = (
    (current_data["Sex_enc"] == 1) * 0.4 +
    (current_data["Pclass"] == 1) * 0.3 +
    np.random.normal(0, 0.1, n_curr)
).clip(0, 1).round().astype(int)

print(f"Reference dataset: {len(reference_data)} rows")
print(f"Current dataset:   {len(current_data)} rows")
print(f"\nDistribution comparison (intentionally drifted features):")
print(f"{'Feature':<15} {'Ref Mean':>10} {'Curr Mean':>10} {'Diff':>8}")
print("-" * 48)
for col in ["Pclass", "Age", "Fare"]:
    ref_mean = reference_data[col].mean()
    curr_mean = current_data[col].mean()
    diff = curr_mean - ref_mean
    print(f"{col:<15} {ref_mean:>10.2f} {curr_mean:>10.2f} {diff:>+8.2f}")

Reference dataset: 500 rows
Current dataset:   200 rows

Distribution comparison (intentionally drifted features):
Feature           Ref Mean  Curr Mean     Diff
------------------------------------------------
Pclass                2.20       1.63    -0.57
Age                  30.11      46.20   +16.09
Fare                 29.74      61.33   +31.59


## 4. Evidently AI — Data Drift Report

In [5]:
import evidently
print(evidently.__version__)

0.7.21


In [9]:
from evidently.sdk.models import PanelMetric
import evidently.sdk as sdk
print(dir(sdk))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'artifacts', 'configs', 'datasets', 'models', 'panels']


In [13]:
from evidently.core.report import Report
from evidently.presets import DataDriftPreset
from evidently import Dataset, DataDefinition

# create datasets
data_definition = DataDefinition(
    numerical_columns=["Age", "Fare", "SibSp", "Parch"],
    categorical_columns=["Pclass", "Sex_enc", "Embarked_enc"],
)

ref_dataset = Dataset.from_pandas(reference_data, data_definition=data_definition)
curr_dataset = Dataset.from_pandas(current_data, data_definition=data_definition)

# run report
report = Report([DataDriftPreset()])
my_report = report.run(reference_data=ref_dataset, current_data=curr_dataset)

print("Report generated successfully!")
print(type(my_report))

Report generated successfully!
<class 'evidently.core.report.Snapshot'>


In [14]:
# extract drift results
snapshot_dict = my_report.dict()

print("=== DRIFT REPORT RESULTS ===\n")

# navigate the snapshot structure
metrics = snapshot_dict.get("metrics", [])
print(f"Total metrics computed: {len(metrics)}")

# print top-level summary
for metric in metrics[:5]:
    metric_id = metric.get("metric_id", {})
    result = metric.get("value", {})
    print(f"\nMetric: {metric_id}")
    if isinstance(result, dict):
        for k, v in list(result.items())[:4]:
            print(f"  {k}: {v}")

=== DRIFT REPORT RESULTS ===

Total metrics computed: 8

Metric: {}
  count: 3.0
  share: 0.42857142857142855

Metric: {}

Metric: {}

Metric: {}

Metric: {}


In [15]:
# explore structure more carefully
import json

snapshot_dict = my_report.dict()
print("Top-level keys:", list(snapshot_dict.keys()))
print("\nFirst metric full structure:")
print(json.dumps(snapshot_dict["metrics"][0], indent=2, default=str)[:800])

Top-level keys: ['metrics', 'tests']

First metric full structure:
{
  "id": "15e89f895b482f9b84ba7274ed18a106",
  "metric_name": "DriftedColumnsCount(drift_share=0.5)",
  "config": {
    "type": "evidently:metric_v2:DriftedColumnsCount",
    "drift_share": 0.5
  },
  "value": {
    "count": 3.0,
    "share": 0.42857142857142855
  }
}


In [16]:
print("=== EVIDENTLY DRIFT REPORT RESULTS ===\n")

for metric in snapshot_dict["metrics"]:
    name = metric.get("metric_name", "unknown")
    value = metric.get("value", {})
    print(f"Metric: {name}")
    if isinstance(value, dict):
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"  value: {value}")
    print()

print("\n=== SUMMARY ===")
# find the drifted columns count metric
for metric in snapshot_dict["metrics"]:
    if "DriftedColumnsCount" in metric.get("metric_name", ""):
        v = metric["value"]
        print(f"Drifted columns: {int(v['count'])} out of 7 ({v['share']:.1%})")
        print(f"Drift detected: {'YES' if v['share'] >= 0.5 else 'PARTIAL'}")

=== EVIDENTLY DRIFT REPORT RESULTS ===

Metric: DriftedColumnsCount(drift_share=0.5)
  count: 3.0
  share: 0.42857142857142855

Metric: ValueDrift(column=Age,method=K-S p_value,threshold=0.05)
  value: 9.08556194878544e-31

Metric: ValueDrift(column=Fare,method=K-S p_value,threshold=0.05)
  value: 1.958801209695936e-09

Metric: ValueDrift(column=SibSp,method=chi-square p_value,threshold=0.05)
  value: 0.05208057998787976

Metric: ValueDrift(column=Parch,method=chi-square p_value,threshold=0.05)
  value: 0.9395452971633225

Metric: ValueDrift(column=Pclass,method=chi-square p_value,threshold=0.05)
  value: 1.891244632637991e-21

Metric: ValueDrift(column=Sex_enc,method=Z-test p_value,threshold=0.05)
  value: 0.29258414027383317

Metric: ValueDrift(column=Embarked_enc,method=chi-square p_value,threshold=0.05)
  value: 0.3559737193501117


=== SUMMARY ===
Drifted columns: 3 out of 7 (42.9%)
Drift detected: PARTIAL


## 5. Alerting Thresholds

In [17]:
def check_drift_alerts(snapshot_dict, thresholds=None):
    if thresholds is None:
        thresholds = {
            "drift_share_warning": 0.2,   # warn if >20% columns drifted
            "drift_share_critical": 0.5,  # critical if >50% columns drifted
            "p_value_threshold": 0.05     # column drifted if p < 0.05
        }

    alerts = []
    column_results = {}

    for metric in snapshot_dict["metrics"]:
        name = metric.get("metric_name", "")
        value = metric.get("value", {})

        if "DriftedColumnsCount" in name:
            share = value.get("share", 0)
            count = int(value.get("count", 0))

            if share >= thresholds["drift_share_critical"]:
                alerts.append({
                    "level": "CRITICAL",
                    "message": f"{count} columns drifted ({share:.1%}) -- retrain model immediately"
                })
            elif share >= thresholds["drift_share_warning"]:
                alerts.append({
                    "level": "WARNING",
                    "message": f"{count} columns drifted ({share:.1%}) -- monitor closely"
                })

        if "ValueDrift" in name:
            p_value = value if not isinstance(value, dict) else value.get("value", 1)
            col = name.split("column=")[1].split(",")[0] if "column=" in name else "unknown"
            drifted = p_value < thresholds["p_value_threshold"]
            column_results[col] = {
                "p_value": p_value,
                "drifted": drifted,
                "status": "DRIFTED" if drifted else "OK"
            }

    print("=== DRIFT ALERTS ===")
    if alerts:
        for alert in alerts:
            print(f"  [{alert['level']}] {alert['message']}")
    else:
        print("  No alerts -- drift within acceptable limits")

    print("\n=== COLUMN STATUS ===")
    print(f"{'Column':<15} {'p-value':>15} {'Status':>10}")
    print("-" * 44)
    for col, result in column_results.items():
        pv = result['p_value']
        pv_str = f"{pv:.2e}" if pv < 0.001 else f"{pv:.4f}"
        print(f"{col:<15} {pv_str:>15} {result['status']:>10}")

    print("\n=== RECOMMENDED ACTION ===")
    drifted_cols = [c for c, r in column_results.items() if r["drifted"]]
    if len(drifted_cols) == 0:
        print("  No action needed -- all features stable")
    elif len(drifted_cols) <= 2:
        print(f"  Monitor closely -- {drifted_cols} showing drift")
        print("  Consider retraining if accuracy drops below threshold")
    else:
        print(f"  Retrain recommended -- {len(drifted_cols)} features drifted: {drifted_cols}")
        print("  Collect fresh labelled data from current distribution")

check_drift_alerts(snapshot_dict)

=== DRIFT ALERTS ===
  [WARNING] 3 columns drifted (42.9%) -- monitor closely

=== COLUMN STATUS ===
Column                  p-value     Status
--------------------------------------------
Age                    9.09e-31    DRIFTED
Fare                   1.96e-09    DRIFTED
SibSp                    0.0521         OK
Parch                    0.9395         OK
Pclass                 1.89e-21    DRIFTED
Sex_enc                  0.2926         OK
Embarked_enc             0.3560         OK

=== RECOMMENDED ACTION ===
  Retrain recommended -- 3 features drifted: ['Age', 'Fare', 'Pclass']
  Collect fresh labelled data from current distribution


## 6. Summary — What We Built Today

| Drift Type | What changes | Detection method | Evidently metric |
|------------|-------------|-----------------|-----------------|
| Data drift | Input feature distributions | Statistical tests (K-S, chi-square, Z-test) | ValueDrift per column |
| Prediction drift | Model output distribution | Same statistical tests on predictions | ValueDrift on target |
| Dataset drift | Overall % of drifted columns | Threshold on drifted column share | DriftedColumnsCount |

**Our Titanic drift simulation results:**
- Age: p=9e-31 → DRIFTED (passengers became 16 years older on average)
- Fare: p=2e-9 → DRIFTED (average fare doubled)
- Pclass: p=2e-21 → DRIFTED (shift to more 1st class passengers)
- SibSp, Parch, Sex_enc, Embarked_enc → NOT drifted

**Alert triggered:** WARNING — 3/7 columns (42.9%) drifted
**Recommended action:** Retrain on fresh data from current distribution

**Statistical tests Evidently uses:**
- Numerical features: Kolmogorov-Smirnov (K-S) test
- Categorical features: Chi-square test or Z-test
- Default p-value threshold: 0.05 (drift detected if p < 0.05)